In [ ]:
!pip install -q --upgrade transformers huggingface_hub datasets

In [ ]:
# HyTE-H: Train Hyperbolic Embedding Projection (Kaggle GPU)
import os
import math
import time
import gc
import struct
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from tqdm.auto import tqdm

# Hardware & Accelerator Setup
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
print(f'Device: {DEVICE} | GPU: {gpu_name}')

if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {vram_gb:.1f} GB')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

Device: cuda | GPU: Tesla T4
VRAM: 15.6 GB


In [ ]:
class C:
    EMBED=384; HIDDEN=256; CURV=1.0; OEM_P=1.0
    BATCH=192; MAX_LEN=128; STEPS=75000; LR=2e-4; WARMUP=1000; TEMP=0.05
    OUT='/kaggle/working/hyte_h_weights'
cfg=C()
os.makedirs(cfg.OUT, exist_ok=True)

In [ ]:
# Frozen BGE-small word embeddings
tokenizer = AutoTokenizer.from_pretrained('BAAI/bge-small-en-v1.5')
bge = AutoModel.from_pretrained('BAAI/bge-small-en-v1.5')
word_embeds = bge.embeddings.word_embeddings.weight.detach().clone()
word_embeds = F.normalize(word_embeds, dim=-1)
word_embeds.requires_grad = False
del bge; gc.collect(); torch.cuda.empty_cache()
print(f'Frozen BGE: {word_embeds.shape[0]}x{word_embeds.shape[1]} = {word_embeds.numel()*4/1e6:.1f} MB')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Frozen BGE: 30522x384 = 46.9 MB


In [ ]:
# HyTE-H model definition - replace the cell that defines class HyTEH

class HyTEH(nn.Module):
    def __init__(self):
        super().__init__()
        self.w1 = nn.Linear(cfg.EMBED, cfg.HIDDEN, bias=False)
        self.w2 = nn.Linear(cfg.HIDDEN, cfg.EMBED, bias=False)
        nn.init.xavier_uniform_(self.w1.weight)
        nn.init.xavier_uniform_(self.w2.weight)
        # Move BGE embeddings to GPU once (shared, frozen)
        self.register_buffer('embeds', word_embeds)

    def add_time(self, x):
        t = torch.sqrt((x ** 2).sum(-1, keepdim=True) + cfg.CURV).clamp(min=1e-6)
        return torch.cat([t, x], dim=-1)

    def oem_pool(self, points, mask):
        t = points[..., 0:1]
        w = t.pow(cfg.OEM_P + 1) * mask.unsqueeze(-1)
        w = w / (w.sum(1, keepdim=True) + 1e-8)
        weighted = (points * w).sum(1)
        inner = -weighted[:, 0:1] ** 2 + (weighted[:, 1:] ** 2).sum(-1, keepdim=True)
        return weighted * torch.sqrt(cfg.CURV / (-inner + 1e-8))

    def forward(self, ids, mask):
        x = self.embeds[ids.clamp(0, self.embeds.shape[0] - 1)]  # embeds now on same device
        x = self.w2(F.relu(self.w1(x))) * mask.unsqueeze(-1)
        return self.oem_pool(self.add_time(x), mask)


model = HyTEH().to(DEVICE)
print(f'Trainable: {sum(p.numel() for p in model.parameters()):,} params')


Trainable: 196,608 params


In [ ]:
# Vectorized pairwise Lorentz similarity: (B,B) matrix
def pairwise_sim(a, b):
    return -a[:, 0:1] * b[:, 0:1].T + a[:, 1:] @ b[:, 1:].T

# Stream MS MARCO
ds = load_dataset('microsoft/ms_marco', 'v1.1', split='train', streaming=True)
print('MS MARCO streaming ready')

MS MARCO streaming ready


In [ ]:
# Replace the training loop cell with this - uses Rust tokenizer (5-10x faster batch tokenization)

import tokenizers as tk

rust_tok = tk.Tokenizer.from_pretrained('BAAI/bge-small-en-v1.5')
rust_tok.enable_padding(pad_id=0, pad_token='[PAD]', length=cfg.MAX_LEN)
rust_tok.enable_truncation(max_length=cfg.MAX_LEN)

try:
    autocast_ctx = torch.amp.autocast('cuda')
except TypeError:
    autocast_ctx = torch.amp.autocast(device_type='cuda')

opt = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=0.01)
scaler = torch.amp.GradScaler('cuda')
model.train()
pbar = tqdm(total=cfg.STEPS, desc=f'B={cfg.BATCH}')

buf_q, buf_d = [], []
step, smooth_loss = 0, 0.0
stream = iter(ds)

while step < cfg.STEPS:
    try: ex = next(stream)
    except StopIteration: stream = iter(ds); ex = next(stream)

    q = ex['query']
    d = ex['passages']['passage_text'][0] if ex.get('passages') else q
    buf_q.append(q); buf_d.append(d)
    if len(buf_q) < cfg.BATCH: continue

    # Rust tokenizer - batch encode is vectorized and fast
    q_enc = rust_tok.encode_batch(buf_q)
    d_enc = rust_tok.encode_batch(buf_d)
    buf_q.clear(); buf_d.clear()

    qi = torch.tensor([e.ids for e in q_enc], device=DEVICE)
    qm = torch.tensor([e.attention_mask for e in q_enc], device=DEVICE)
    di = torch.tensor([e.ids for e in d_enc], device=DEVICE)
    dm = torch.tensor([e.attention_mask for e in d_enc], device=DEVICE)

    with autocast_ctx:
        qe, de = model(qi, qm), model(di, dm)
        sim = pairwise_sim(qe, de) / cfg.TEMP
        loss = F.cross_entropy(sim, torch.arange(cfg.BATCH, device=DEVICE))

    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt)
    scaler.update()

    if step < cfg.WARMUP:
        lr = cfg.LR * step / cfg.WARMUP
    else:
        prog = (step-cfg.WARMUP)/(cfg.STEPS-cfg.WARMUP)
        lr = cfg.LR * 0.5*(1+math.cos(math.pi*prog))
    for pg in opt.param_groups: pg['lr'] = lr

    smooth_loss = 0.9*smooth_loss + 0.1*loss.item()
    step += 1
    if step % 100 == 0:
        pbar.update(100)
        pbar.set_description(f'Loss {smooth_loss:.4f} | LR {lr:.2e}')

pbar.close()
print(f'Done: {step} steps, loss={smooth_loss:.4f}')


B=192:   0%|          | 0/75000 [00:00<?, ?it/s]

Done: 75000 steps, loss=0.5666


In [ ]:
# Export C++ binaries
for name, layer in [('proj_weight', model.w1), ('proj_out', model.w2)]:
    arr = layer.weight.detach().cpu().float().numpy()
    with open(f'{cfg.OUT}/{name}.bin', 'wb') as f:
        f.write(struct.pack('<ii', arr.shape[0], arr.shape[1]))
        f.write(arr.tobytes())
    print(f'{name}.bin: {arr.shape} = {arr.nbytes/1024:.1f} KB')

import json
json.dump({'embed':cfg.EMBED, 'hidden':cfg.HIDDEN, 'curv':cfg.CURV, 'oem_p':cfg.OEM_P, 'steps':step},
         open(f'{cfg.OUT}/config.json','w'), indent=2)
print(f'Saved to {cfg.OUT}/')

proj_weight.bin: (256, 384) = 384.0 KB
proj_out.bin: (384, 256) = 384.0 KB
Saved to /kaggle/working/hyte_h_weights/


In [ ]:
# Verify radial hierarchy
model.eval()
pairs = [('animal','mammal'),('mammal','dog'),('science','physics'),('disease','cancer')]
with torch.no_grad():
    for g,s in pairs:
        gt = tokenizer(g, return_tensors='pt', padding='max_length', max_length=32, truncation=True)
        st = tokenizer(s, return_tensors='pt', padding='max_length', max_length=32, truncation=True)
        ge = model(gt['input_ids'].to(DEVICE), gt['attention_mask'].to(DEVICE))
        se = model(st['input_ids'].to(DEVICE), st['attention_mask'].to(DEVICE))
        pct = (se[0,0].item()/ge[0,0].item()-1)*100
        arrow = 'SPECIFIC!' if pct > 0 else 'collapsed'
        print(f'{g:12s} -> {s:12s}  r={ge[0,0].item():.3f}->{se[0,0].item():.3f}  {arrow} ({pct:+.1f}%)')

animal       -> mammal        r=7.459->14.704  SPECIFIC! (+97.1%)
mammal       -> dog           r=14.704->11.482  collapsed (-21.9%)
science      -> physics       r=4.022->7.502  SPECIFIC! (+86.5%)
disease      -> cancer        r=8.787->9.732  SPECIFIC! (+10.8%)
